In [16]:
# Neural Identifier Training - 2-DOF Planar Manipulator
# Methods: EKF, UKF, Particle Filter
# Con características específicas por neurona

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# 1) True nonlinear system (2-DOF Robot Arm)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a 2-link planar robot arm.
    state = [q1, q2, dq1, dq2] (Angles and Velocities)
    u     = [tau1, tau2]       (Torques)
    """
    # Robot Parameters
    m1, m2 = 1.0, 1.0  # Mass (kg)
    l1, l2 = 1.0, 1.0  # Lengths (m)
    g = 9.81
    
    q1, q2, dq1, dq2 = state
    tau1, tau2 = u

    # --- Mass Matrix M(q) ---
    c2 = np.cos(q2)
    s2 = np.sin(q2)
    
    M11 = (m1 + m2) * l1**2 + m2 * l2**2 + 2 * m2 * l1 * l2 * c2
    M12 = m2 * l2**2 + m2 * l1 * l2 * c2
    M21 = M12
    M22 = m2 * l2**2
    M = np.array([[M11, M12], [M21, M22]])

    # --- Coriolis/Centrifugal Matrix C(q, dq) ---
    h = -m2 * l1 * l2 * s2
    C11 = h * dq2
    C12 = h * (dq1 + dq2)
    C21 = -h * dq1
    C22 = 0.0
    C = np.array([[C11, C12], [C21, C22]])

    # --- Gravity Vector G(q) ---
    s1 = np.sin(q1)
    s12 = np.sin(q1 + q2)
    G1 = (m1 + m2) * g * l1 * s1 + m2 * g * l2 * s12
    G2 = m2 * g * l2 * s12
    G = np.array([G1, G2])

    # --- Equation of Motion: M*ddq + C*dq + G = tau ---
    damping = 0.5 * np.array([dq1, dq2])
    torque_vector = np.array([tau1, tau2])
    
    rhs = torque_vector - (C @ np.array([dq1, dq2])) - G - damping
    
    # Solve for accelerations
    ddq = np.linalg.solve(M, rhs)
    
    return np.concatenate(([dq1, dq2], ddq))

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-4):
    """
    Euler integration step with Gaussian process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Add Gaussian process noise
    noise = np.random.randn(4) * process_noise_std
    return x_kp1 + noise

# ============================================================
# 2) RHONN structure - CARACTERÍSTICAS POR NEURONA
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z). Beta reduced to widen the active range."""
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input, neuron_index):
    """
    Features for 2-DOF Arm - ESPECÍFICAS PARA CADA NEURONA.
    
    x_est = [q1, q2, dq1, dq2]
    u_input = [tau1, tau2]
    neuron_index: índice de la neurona (0=q1, 1=q2, 2=dq1, 3=dq2)
    
    Cada neurona puede tener su propia estructura de características.
    """
    q1, q2, dq1, dq2 = x_est
    tau1, tau2 = u_input
    
    # Términos básicos sigmoidales
    s_q1 = sigmoidal(q1)
    s_q2 = sigmoidal(q2)
    s_dq1 = sigmoidal(dq1)
    s_dq2 = sigmoidal(dq2)
    
    # Términos trigonométricos (útiles para dinámica de robots)
    sin_q1 = np.sin(q1)
    sin_q2 = np.sin(q2)
    cos_q1 = np.cos(q1)
    cos_q2 = np.cos(q2)
    sin_q1q2 = np.sin(q1 + q2)
    
    # ========== CARACTERÍSTICAS ESPECÍFICAS POR NEURONA ==========
    
    if neuron_index == 0:  # Neurona para q1 (ángulo articulación 1)
        return np.array([
            # s_q1,                      # Estado actual
            # s_dq1,                     # Velocidad actual
            s_q1 * s_dq1,             # Interacción ángulo-velocidad
            s_q1**2,                   # Término cuadrático
            s_q2,                      # Acoplamiento con q2
            # sin_q1,                    # Término gravitacional
            # tau1 * 0.1,               # Entrada de control
            1.0                        # Bias
        ])
    
    elif neuron_index == 1:  # Neurona para q2 (ángulo articulación 2)
        return np.array([
            # s_q2,                      # Estado actual
            # s_dq2,                     # Velocidad actual
            s_q2 * s_dq2,             # Interacción ángulo-velocidad
            s_q2**3,                   # Término cúbico (mayor no linealidad)
            s_q1 * s_q2,              # Acoplamiento con q1
            # sin_q2,                    # Término gravitacional
            # sin_q1q2,                  # Acoplamiento cinemático
            tau2 * 0.1,               # Entrada de control
            # 1.0                        # Bias
        ])
    
    elif neuron_index == 2:  # Neurona para dq1 (velocidad articulación 1)
        return np.array([
            # s_dq1,                     # Estado actual
            s_dq1**2,                  # Término cuadrático (fricción)
            # s_q1,                      # Dependencia del ángulo
            s_q2 * s_dq1,             # Coriolis proxy
            s_q2 * s_dq2,             # Coriolis proxy
            # s_dq2,                     # Acoplamiento velocidades
            # cos_q2,                    # Términos de masa variable
            # tau1 * 0.1,               # Entrada de control (lineal)
            tau2 * 0.05,              # Acoplamiento de torques
            1.0                        # Bias
        ])
    
    elif neuron_index == 3:  # Neurona for dq2 (velocidad articulación 2)
        return np.array([
            # s_dq2,                     # Estado actual
            s_dq2**2,                  # Término cuadrático (fricción)
            # s_q2,                      # Dependencia del ángulo
            s_q1 * s_dq2,             # Coriolis proxy
            s_dq1 * s_dq2,            # Interacción velocidades
            # s_dq1,                     # Acoplamiento con dq1
            # cos_q2,                    # Términos de masa variable
            tau2 * 0.1,               # Entrada de control (lineal)
            tau1 * 0.05,              # Acoplamiento de torques
            # 1.0                        # Bias
        ])
    
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# Función auxiliar para obtener el tamaño de características de cada neurona
def get_z_size(neuron_index):
    """Retorna el número de características para una neurona dada."""
    if neuron_index == 0:
        return 4
    elif neuron_index == 1:
        return 4
    elif neuron_index == 2:
        return 5
    elif neuron_index == 3:
        return 5
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# ============================================================
# 3) Trainers (EKF, UKF, PF) - ADAPTADOS PARA MÚLTIPLES TAMAÑOS
# ============================================================

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k, neuron_idx):
        z = construct_z_vector(x_k, u_k, neuron_idx)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, P0=1.0, Q=1e-4, R=1e-5):
        self.n_neurons = n_neurons
        # Cada neurona tiene su propio número de pesos
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i))*P0 for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*Q for i in range(n_neurons)]
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            H = z.reshape(-1, 1)
            
            # Predict P
            P_pred = self.P[i] + self.Q_matrices[i]
            
            # Kalman Gain
            S = self.R + (H.T @ P_pred @ H)[0,0]
            K = (P_pred @ H).flatten() / S
            
            # Error
            y_pred = np.dot(self.weights[i], z)
            err = x_kp1[i] - y_pred
            
            # Update Weights
            self.weights[i] += self.eta * K * err
            
            # Update Covariance (Joseph form)
            I_KH = np.eye(len(z)) - np.outer(K, z)
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K, K)*self.R

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, alpha=1e-2):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i)) for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*1e-4 for i in range(n_neurons)]
        self.R = 1e-5
        self.eta = eta
        self.alpha = alpha

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n = get_z_size(i)
            
            # Sigma params (calculados para cada neurona)
            lambda_ = self.alpha**2 * n - n
            Wm = np.full(2*n+1, 1/(2*(n+lambda_)))
            Wc = np.copy(Wm)
            Wm[0] = lambda_/(n+lambda_)
            Wc[0] = Wm[0] + (3 - self.alpha**2)
            
            # Generate Sigmas
            try:
                L = np.linalg.cholesky((n + lambda_) * self.P[i])
            except:
                L = np.eye(n) * 0.1
                
            sigmas = np.zeros((2*n+1, n))
            sigmas[0] = self.weights[i]
            for k in range(n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[n+k+1] = self.weights[i] - L[:,k]
            
            # Transform
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(Wm * Y_sigmas)
            
            # Covariances
            Py = np.sum(Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(n)
            for k in range(2*n+1):
                Pxy += Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
                
            # Update
            K = Pxy / Py
            err = x_kp1[i] - y_mean
            self.weights[i] += self.eta * K * err
            self.P[i] -= np.outer(K, K) * Py
            
            # Regularize P
            self.P[i] += np.eye(n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, n_particles=1000, Q_std=None, R_std=None):
        self.n_neurons = n_neurons
        self.n_particles = n_particles
        # Cada neurona tiene partículas de diferente dimensión
        self.particles = [np.random.randn(n_particles, get_z_size(i))*0.2 
                         for i in range(n_neurons)]
        self.weights_pf = [np.ones(n_particles)/n_particles for _ in range(n_neurons)]
        
        # Q_std y R_std por neurona (permite ajuste fino por estado)
        if Q_std is None:
            self.Q_std = [0.5] * n_neurons  # Default: mismo valor para todas
        else:
            self.Q_std = Q_std if isinstance(Q_std, list) else [Q_std] * n_neurons
            
        if R_std is None:
            self.R_std = [0.2] * n_neurons  # Default: mismo valor para todas
        else:
            self.R_std = R_std if isinstance(R_std, list) else [R_std] * n_neurons

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n_weights = get_z_size(i)
            
            # 1. Drift (con Q_std específico por neurona)
            self.particles[i] += np.random.randn(self.n_particles, n_weights) * self.Q_std[i]
            
            # 2. Weight (con R_std específico por neurona)
            preds = self.particles[i] @ z
            err = x_kp1[i] - preds
            likelihood = np.exp(-0.5 * (err/self.R_std[i])**2)
            self.weights_pf[i] *= (likelihood + 1e-300)
            self.weights_pf[i] /= np.sum(self.weights_pf[i])
            
            # 3. Resample
            eff_N = 1.0 / np.sum(self.weights_pf[i]**2)
            if eff_N < self.n_particles/2:
                indices = np.random.choice(self.n_particles, self.n_particles, 
                                         p=self.weights_pf[i])
                self.particles[i] = self.particles[i][indices]
                self.weights_pf[i].fill(1.0/self.n_particles)
                
    def get_estimates(self):
        return [np.average(self.particles[i], axis=0, weights=self.weights_pf[i]) 
                for i in range(self.n_neurons)]

# ============================================================
# 4) Simulation Main Loop
# ============================================================
if __name__ == "__main__":
    np.random.seed(7517)
    n_steps = 1000
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    
    n_states = 4
    
    # Noise parameters
    process_noise_std = 1e-2
    measurement_noise_std = 1e-3  # Measurement noise standard deviation

    
    # Init Trainers
    ekf = EKF_Trainer(n_states, eta=1.0)
    ukf = UKF_Trainer(n_states, eta=0.9)
    pf = PF_Trainer(4, n_particles=1200, 
                # Q_std=[0.3, 0.01, 0.6, 0.5],  # Different Q per neuron
                # R_std=[0.01, 0.02, 0.03, 0.02])  # Different R per neuron
                Q_std=1,
                R_std=1e-6
    )

    
    # Arrays
    x_true = np.zeros((n_steps, 4))
    x_measured = np.zeros((n_steps, 4))  # Noisy measurements
    x_est_ekf = np.zeros((n_steps, 4))
    x_est_ukf = np.zeros((n_steps, 4))
    x_est_pf = np.zeros((n_steps, 4))
    
    # Initial Conditions
    x_true[0] = [-np.pi/2, 0, 0, 0] 
    x_measured[0] = x_true[0]  # No noise at t=0
    x_est_ekf[0] = x_measured[0]
    x_est_ukf[0] = x_measured[0]
    x_est_pf[0]  = x_measured[0]
    
    # Excitation Input
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        tau1 = 30.0 * np.sin(2.0 * t[k]) 
        tau2 = 15.0 * np.cos(3.0 * t[k])
        u_hist[k] = [tau1, tau2]

    print("Simulating 2-DOF Manipulator with neuron-specific features...")
    print("\nEstructura de características por neurona:")
    for i in range(n_states):
        print(f"  Neurona {i}: {get_z_size(i)} características")
    print(f"\nNivel de ruido:")
    print(f"  Proceso: σ = {process_noise_std:.2e}")
    print(f"  Medición: σ = {measurement_noise_std:.2e}")
    
    for k in range(n_steps - 1):
        # 1. Physics Step (with Gaussian process noise)
        x_true[k+1] = plant(x_true[k], u_hist[k], dt, process_noise_std)
        
        # 2. Add Gaussian measurement noise
        x_measured[k+1] = x_true[k+1] + np.random.randn(4) * measurement_noise_std
        
        # 3. Identify (using noisy measurements)
        # EKF
        ekf.update(x_measured[k+1], x_measured[k], u_hist[k])
        for i in range(4): 
            z = construct_z_vector(x_measured[k], u_hist[k], i)
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z)
            
        # UKF
        ukf.update(x_measured[k+1], x_measured[k], u_hist[k])
        for i in range(4): 
            z = construct_z_vector(x_measured[k], u_hist[k], i)
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z)
            
        # PF
        pf.update(x_measured[k+1], x_measured[k], u_hist[k])
        w_pf = pf.get_estimates()
        for i in range(4): 
            z = construct_z_vector(x_measured[k], u_hist[k], i)
            x_est_pf[k+1, i] = np.dot(w_pf[i], z)
            
        if k % 100 == 0: print(f"Step {k}")

    # ============================================================
    # 5) Métricas de Error
    # ============================================================
    
    print("\n" + "="*70)
    print("CÁLCULO DE MÉTRICAS DE ERROR")
    print("="*70)
    
    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 12,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }
    
    # Nombres de estados para reportes
    state_names = ['q₁', 'q₂', 'dq₁', 'dq₂']
    
    # Diccionarios para almacenar métricas
    metrics = {
        'MSE': {'EKF': [], 'UKF': [], 'PF': []},
        'RMSE': {'EKF': [], 'UKF': [], 'PF': []},
        'MAE': {'EKF': [], 'UKF': [], 'PF': []},
        'NRMSE': {'EKF': [], 'UKF': [], 'PF': []}
    }
    
    # Calcular métricas para cada estado
    for i in range(4):
        # Errores
        err_ekf = x_true[:, i] - x_est_ekf[:, i]
        err_ukf = x_true[:, i] - x_est_ukf[:, i]
        err_pf = x_true[:, i] - x_est_pf[:, i]
        
        # MSE
        mse_ekf = np.mean(err_ekf**2)
        mse_ukf = np.mean(err_ukf**2)
        mse_pf = np.mean(err_pf**2)
        
        # RMSE
        rmse_ekf = np.sqrt(mse_ekf)
        rmse_ukf = np.sqrt(mse_ukf)
        rmse_pf = np.sqrt(mse_pf)
        
        # MAE
        mae_ekf = np.mean(np.abs(err_ekf))
        mae_ukf = np.mean(np.abs(err_ukf))
        mae_pf = np.mean(np.abs(err_pf))
        
        # NRMSE (normalized by range)
        range_true = np.max(x_true[:, i]) - np.min(x_true[:, i])
        nrmse_ekf = rmse_ekf / range_true if range_true > 0 else 0
        nrmse_ukf = rmse_ukf / range_true if range_true > 0 else 0
        nrmse_pf = rmse_pf / range_true if range_true > 0 else 0
        
        # Almacenar
        metrics['MSE']['EKF'].append(mse_ekf)
        metrics['MSE']['UKF'].append(mse_ukf)
        metrics['MSE']['PF'].append(mse_pf)
        
        metrics['RMSE']['EKF'].append(rmse_ekf)
        metrics['RMSE']['UKF'].append(rmse_ukf)
        metrics['RMSE']['PF'].append(rmse_pf)
        
        metrics['MAE']['EKF'].append(mae_ekf)
        metrics['MAE']['UKF'].append(mae_ukf)
        metrics['MAE']['PF'].append(mae_pf)
        
        metrics['NRMSE']['EKF'].append(nrmse_ekf)
        metrics['NRMSE']['UKF'].append(nrmse_ukf)
        metrics['NRMSE']['PF'].append(nrmse_pf)
    
    # Métricas totales (suma de todos los estados)
    total_metrics = {}
    for metric_name in ['MSE', 'RMSE', 'MAE', 'NRMSE']:
        total_metrics[metric_name] = {
            'EKF': np.sum(metrics[metric_name]['EKF']),
            'UKF': np.sum(metrics[metric_name]['UKF']),
            'PF': np.sum(metrics[metric_name]['PF'])
        }
    
    # Imprimir resultados
    print("\n--- MÉTRICAS POR ESTADO ---")
    for i, state_name in enumerate(state_names):
        print(f"\n{state_name}:")
        print(f"  MSE:   EKF={metrics['MSE']['EKF'][i]:.6f}  UKF={metrics['MSE']['UKF'][i]:.6f}  PF={metrics['MSE']['PF'][i]:.6f}")
        print(f"  RMSE:  EKF={metrics['RMSE']['EKF'][i]:.6f}  UKF={metrics['RMSE']['UKF'][i]:.6f}  PF={metrics['RMSE']['PF'][i]:.6f}")
        print(f"  MAE:   EKF={metrics['MAE']['EKF'][i]:.6f}  UKF={metrics['MAE']['UKF'][i]:.6f}  PF={metrics['MAE']['PF'][i]:.6f}")
        print(f"  NRMSE: EKF={metrics['NRMSE']['EKF'][i]:.4f}  UKF={metrics['NRMSE']['UKF'][i]:.4f}  PF={metrics['NRMSE']['PF'][i]:.4f}")
    
    print("\n--- MÉTRICAS TOTALES (SUMA) ---")
    for metric_name in ['MSE', 'RMSE', 'MAE', 'NRMSE']:
        print(f"\n{metric_name}:")
        print(f"  EKF: {total_metrics[metric_name]['EKF']:.6f}")
        print(f"  UKF: {total_metrics[metric_name]['UKF']:.6f}")
        print(f"  PF:  {total_metrics[metric_name]['PF']:.6f}")
        
        # Determinar mejor filtro
        best_filter = min(total_metrics[metric_name], key=total_metrics[metric_name].get)
        print(f"  🏆 MEJOR: {best_filter}")
    
    # ============================================================
    # 6) Visualización - Formato Tesis
    # ============================================================
    
    print("\nGenerando visualizaciones...")
    
    # --- Gráficas por Estado ---
    states_info = [
        {'idx': 0, 'var': 'q₁', 'desc': 'Ángulo Articulación 1', 'y_label': 'Ángulo q₁ (rad)'},
        {'idx': 1, 'var': 'q₂', 'desc': 'Ángulo Articulación 2', 'y_label': 'Ángulo q₂ (rad)'},
        {'idx': 2, 'var': 'dq₁', 'desc': 'Velocidad Articulación 1', 'y_label': 'Velocidad dq₁ (rad/s)'},
        {'idx': 3, 'var': 'dq₂', 'desc': 'Velocidad Articulación 2', 'y_label': 'Velocidad dq₂ (rad/s)'}
    ]
    
    for state_info in states_info:
        i = state_info['idx']
        
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=t, y=x_true[:, i],
            mode='lines',
            name='Estado Real',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ekf[:, i],
            mode='lines',
            name='EKF-RHONN',
            line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
            showlegend=True
        ))
        
        fig.add_trace(go.Scatter(
            x=t, y=x_est_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))
        
        fig.update_layout(
            title={
                'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Manipulador 2-DOF',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Tiempo (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.02,
                y=0.98,
                xanchor='left',
                yanchor='top',
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=80, b=60)
        )
        
        fig.show()
    
    # --- Gráficas de Métricas (RMSE, MAE, NRMSE) ---
    
    # RMSE por estado
    fig_rmse = go.Figure()
    
    x_pos = np.arange(len(state_names))
    width = 0.25
    
    fig_rmse.add_trace(go.Bar(
        name='EKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['RMSE']['EKF'],
        marker_color='#1f77b4',
        text=[f'{v:.4f}' for v in metrics['RMSE']['EKF']],
        textposition='outside'
    ))
    
    fig_rmse.add_trace(go.Bar(
        name='UKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['RMSE']['UKF'],
        marker_color='#2ca02c',
        text=[f'{v:.4f}' for v in metrics['RMSE']['UKF']],
        textposition='outside'
    ))
    
    fig_rmse.add_trace(go.Bar(
        name='PF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['RMSE']['PF'],
        marker_color='#d62728',
        text=[f'{v:.4f}' for v in metrics['RMSE']['PF']],
        textposition='outside'
    ))
    
    fig_rmse.update_layout(
        title={
            'text': 'RMSE por Estado - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Estado',
        yaxis_title='RMSE',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_rmse.show()
    
    # MAE por estado
    fig_mae = go.Figure()
    
    fig_mae.add_trace(go.Bar(
        name='EKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['MAE']['EKF'],
        marker_color='#1f77b4',
        text=[f'{v:.4f}' for v in metrics['MAE']['EKF']],
        textposition='outside'
    ))
    
    fig_mae.add_trace(go.Bar(
        name='UKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['MAE']['UKF'],
        marker_color='#2ca02c',
        text=[f'{v:.4f}' for v in metrics['MAE']['UKF']],
        textposition='outside'
    ))
    
    fig_mae.add_trace(go.Bar(
        name='PF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['MAE']['PF'],
        marker_color='#d62728',
        text=[f'{v:.4f}' for v in metrics['MAE']['PF']],
        textposition='outside'
    ))
    
    fig_mae.update_layout(
        title={
            'text': 'MAE por Estado - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Estado',
        yaxis_title='MAE',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_mae.show()
    
    # NRMSE por estado
    fig_nrmse = go.Figure()
    
    fig_nrmse.add_trace(go.Bar(
        name='EKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['NRMSE']['EKF'],
        marker_color='#1f77b4',
        text=[f'{v:.4f}' for v in metrics['NRMSE']['EKF']],
        textposition='outside'
    ))
    
    fig_nrmse.add_trace(go.Bar(
        name='UKF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['NRMSE']['UKF'],
        marker_color='#2ca02c',
        text=[f'{v:.4f}' for v in metrics['NRMSE']['UKF']],
        textposition='outside'
    ))
    
    fig_nrmse.add_trace(go.Bar(
        name='PF-RHONN',
        x=[s + ' ' for s in state_names],
        y=metrics['NRMSE']['PF'],
        marker_color='#d62728',
        text=[f'{v:.4f}' for v in metrics['NRMSE']['PF']],
        textposition='outside'
    ))
    
    fig_nrmse.update_layout(
        title={
            'text': 'NRMSE por Estado - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Estado',
        yaxis_title='NRMSE',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_nrmse.show()
    
    # --- Comparación de métricas totales ---
    fig_total = go.Figure()
    
    filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
    
    fig_total.add_trace(go.Bar(
        name='RMSE Total',
        x=filters,
        y=[total_metrics['RMSE']['EKF'], total_metrics['RMSE']['UKF'], total_metrics['RMSE']['PF']],
        marker_color='#636EFA',
        text=[f'{v:.4f}' for v in [total_metrics['RMSE']['EKF'], total_metrics['RMSE']['UKF'], total_metrics['RMSE']['PF']]],
        textposition='outside'
    ))
    
    fig_total.add_trace(go.Bar(
        name='MAE Total',
        x=filters,
        y=[total_metrics['MAE']['EKF'], total_metrics['MAE']['UKF'], total_metrics['MAE']['PF']],
        marker_color='#EF553B',
        text=[f'{v:.4f}' for v in [total_metrics['MAE']['EKF'], total_metrics['MAE']['UKF'], total_metrics['MAE']['PF']]],
        textposition='outside'
    ))
    
    fig_total.add_trace(go.Bar(
        name='NRMSE Total',
        x=filters,
        y=[total_metrics['NRMSE']['EKF'], total_metrics['NRMSE']['UKF'], total_metrics['NRMSE']['PF']],
        marker_color='#00CC96',
        text=[f'{v:.4f}' for v in [total_metrics['NRMSE']['EKF'], total_metrics['NRMSE']['UKF'], total_metrics['NRMSE']['PF']]],
        textposition='outside'
    ))
    
    fig_total.update_layout(
        title={
            'text': 'Comparación de Métricas Totales - Manipulador 2-DOF',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Valor de Métrica',
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=100, b=60)
    )
    
    fig_total.show()
    
    print("\n✅ Visualización completa con RMSE, MAE y NRMSE.")


Simulating 2-DOF Manipulator with neuron-specific features...

Estructura de características por neurona:
  Neurona 0: 4 características
  Neurona 1: 4 características
  Neurona 2: 5 características
  Neurona 3: 5 características

Nivel de ruido:
  Proceso: σ = 1.00e-02
  Medición: σ = 1.00e-03
Step 0
Step 100
Step 200
Step 300
Step 400
Step 500
Step 600
Step 700
Step 800
Step 900

CÁLCULO DE MÉTRICAS DE ERROR

--- MÉTRICAS POR ESTADO ---

q₁:
  MSE:   EKF=0.000004  UKF=0.006770  PF=22.636046
  RMSE:  EKF=0.001986  UKF=0.082281  PF=4.757735
  MAE:   EKF=0.001598  UKF=0.066360  PF=3.851516
  NRMSE: EKF=0.0001  UKF=0.0050  PF=0.2883

q₂:
  MSE:   EKF=0.015035  UKF=0.473741  PF=851.808484
  RMSE:  EKF=0.122616  UKF=0.688288  PF=29.185758
  MAE:   EKF=0.032773  UKF=0.277326  PF=19.527280
  NRMSE: EKF=0.0044  UKF=0.0245  PF=1.0402

dq₁:
  MSE:   EKF=0.000170  UKF=0.230244  PF=27.497886
  RMSE:  EKF=0.013043  UKF=0.479838  PF=5.243843
  MAE:   EKF=0.008016  UKF=0.313842  PF=4.044341
  NRMSE:


✅ Visualización completa con RMSE, MAE y NRMSE.


In [17]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# ==========================================
# 1. Configuration & Constants
# ==========================================
DT = 0.05
STEPS = 1000
PROCESS_NOISE_STD = 0.1  # Laplacian Scale
NUM_PARTICLES = 200
NUM_NEURONS = 3          # px, py, theta
NUM_FEATURES = 19        # Dimension of RHONN input z
NUM_MONTE_CARLO = 50     # Monte Carlo runs for statistical analysis

# ==========================================
# 2. Math & RHONN Utils
# ==========================================
def sigmoid(z, beta=1.0):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_state, u_input):
    s_px = sigmoid(x_state[0])
    s_py = sigmoid(x_state[1])
    s_th = sigmoid(x_state[2])
    s_v  = sigmoid(u_input[0])
    s_w  = sigmoid(u_input[1])
    
    return np.array([
        s_px, s_py, s_th, s_v, s_w,
        s_px*s_py, s_px*s_th, s_py*s_th,
        s_px*s_v, s_py*s_v, s_th*s_v, s_th*s_w, s_v*s_w,
        s_px**2, s_py**2, s_th**2, s_v**2, s_w**2,
        1.0
    ])

def rhonn_predict(z, weights):
    return np.dot(weights, z)

# ==========================================
# 3. Plant Dynamics (True System)
# ==========================================
def plant_step(x_k, u_k, dt, noise_std):
    delta_v = np.random.laplace(0, noise_std * 2)
    delta_w = np.random.laplace(0, noise_std)
    
    noise_px = np.random.laplace(0, noise_std / 10)
    noise_py = np.random.laplace(0, noise_std / 10)
    noise_th = np.random.laplace(0, noise_std / 10)
    
    v_actual = u_k[0] + delta_v
    w_actual = u_k[1] + delta_w
    
    px_dot = v_actual * np.cos(x_k[2])
    py_dot = v_actual * np.sin(x_k[2])
    theta_dot = w_actual
    
    x_kp1 = np.zeros(3)
    x_kp1[0] = x_k[0] + dt * px_dot + noise_px
    x_kp1[1] = x_k[1] + dt * py_dot + noise_py
    x_kp1[2] = x_k[2] + dt * theta_dot + noise_th
    
    return x_kp1

# ==========================================
# 4. EKF Trainer
# ==========================================
class EKF_RHONN_Trainer:
    def __init__(self, num_neurons, num_features, eta=1.0, initial_weights=None):
        self.num_neurons = num_neurons
        self.eta = eta
        
        if initial_weights is not None:
            self.weights = np.array(initial_weights, dtype=float)
        else:
            self.weights = np.random.randn(num_neurons, num_features) * 0.1

        self.P = [np.eye(num_features) * 1.0 for _ in range(num_neurons)]
        self.Q = [np.eye(num_features) * 1e-4 for _ in range(num_neurons)]
        self.R = [0.01 for _ in range(num_neurons)]

    def update(self, chi_kp1, chi_k, u_k, x_hat_prev):
        x_state_z = np.array([chi_k[0], chi_k[1], x_hat_prev[2]])
        z = construct_z_vector(x_state_z, u_k)
        
        for i in range(self.num_neurons):
            P_pred = self.P[i] + self.Q[i]
            Pz = P_pred @ z
            M = self.R[i] + z @ Pz
            K = Pz / (M + 1e-12)
            
            prediction = np.dot(self.weights[i], z)
            error = chi_kp1[i] - prediction
            
            self.weights[i] = self.weights[i] + self.eta * K * error
            P_update = P_pred - np.outer(K, Pz)
            self.P[i] = 0.5 * (P_update + P_update.T)

    def get_estimate(self, chi_k, x_hat_prev, u_k):
        x_state_z = np.array([chi_k[0], chi_k[1], x_hat_prev[2]])
        z = construct_z_vector(x_state_z, u_k)
        return np.array([rhonn_predict(z, self.weights[i]) for i in range(3)])

# ==========================================
# 5. UKF Trainer (CORRECTED)
# ==========================================
class UKF_RHONN_Trainer:
    def __init__(self, num_neurons, num_features, eta=1.0, initial_weights=None):
        self.num_neurons = num_neurons
        self.num_features = num_features
        self.eta = eta
        
        if initial_weights is not None:
            self.weights = np.array(initial_weights, dtype=float)
        else:
            self.weights = np.random.randn(num_neurons, num_features) * 0.1
        
        self.P = [np.eye(num_features) * 1.0 for _ in range(num_neurons)]
        self.Q = [np.eye(num_features) * 1e-4 for _ in range(num_neurons)]
        self.R = [0.01 for _ in range(num_neurons)]
        
        # UKF Parameters (Van der Merwe scaled sigma points)
        self.alpha = 1e-3  # Spread of sigma points
        self.beta = 2.0    # Prior knowledge (2 = Gaussian optimal)
        self.kappa = 0.0   # Secondary scaling parameter
        
        # Compute lambda
        self.lam = self.alpha**2 * (num_features + self.kappa) - num_features
        
        # Weights for sigma points
        self.n_sigma = 2 * num_features + 1
        self.Wm = np.zeros(self.n_sigma)  # Mean weights
        self.Wc = np.zeros(self.n_sigma)  # Covariance weights
        
        self.Wm[0] = self.lam / (num_features + self.lam)
        self.Wc[0] = self.lam / (num_features + self.lam) + (1 - self.alpha**2 + self.beta)
        
        for i in range(1, self.n_sigma):
            self.Wm[i] = 1.0 / (2.0 * (num_features + self.lam))
            self.Wc[i] = 1.0 / (2.0 * (num_features + self.lam))
    
    def generate_sigma_points(self, mean, P):
        """Generate sigma points using Cholesky decomposition"""
        n = len(mean)
        
        # Ensure P is positive definite
        P_safe = 0.5 * (P + P.T)
        P_safe += np.eye(n) * 1e-9
        
        try:
            L = np.linalg.cholesky((n + self.lam) * P_safe)
        except np.linalg.LinAlgError:
            # Fallback to eigenvalue decomposition
            eigval, eigvec = np.linalg.eigh(P_safe)
            eigval = np.maximum(eigval, 1e-9)
            L = eigvec @ np.diag(np.sqrt(eigval * (n + self.lam)))
        
        sigma_points = np.zeros((self.n_sigma, n))
        sigma_points[0] = mean
        
        for i in range(n):
            sigma_points[i + 1] = mean + L[:, i]
            sigma_points[n + i + 1] = mean - L[:, i]
        
        return sigma_points
    
    def measurement_function(self, weights, z):
        """
        Measurement function: h(w) = w^T * z
        This is the RHONN output for given weights
        """
        return np.dot(weights, z)
    
    def update(self, chi_kp1, chi_k, u_k, x_hat_prev):
        # Construct observation vector z (same for all neurons)
        x_state_z = np.array([chi_k[0], chi_k[1], x_hat_prev[2]])
        z = construct_z_vector(x_state_z, u_k)
        
        for i in range(self.num_neurons):
            # 1. Generate sigma points in weight space
            sigma_points = self.generate_sigma_points(self.weights[i], self.P[i])
            
            # 2. Predict step (weights evolve with process noise)
            # For weight space: w_k+1 = w_k + noise
            # Predicted mean
            w_pred = np.sum(self.Wm[:, np.newaxis] * sigma_points, axis=0)
            
            # Predicted covariance
            diff_pred = sigma_points - w_pred
            P_pred = np.zeros((self.num_features, self.num_features))
            for j in range(self.n_sigma):
                P_pred += self.Wc[j] * np.outer(diff_pred[j], diff_pred[j])
            P_pred += self.Q[i]
            
            # 3. Regenerate sigma points with predicted covariance
            sigma_points_pred = self.generate_sigma_points(w_pred, P_pred)
            
            # 4. Transform sigma points through measurement function
            # For each sigma point (weight vector), compute y = w^T * z
            y_sigma = np.zeros(self.n_sigma)
            for j in range(self.n_sigma):
                y_sigma[j] = self.measurement_function(sigma_points_pred[j], z)
            
            # 5. Predicted measurement mean
            y_pred = np.sum(self.Wm * y_sigma)
            
            # 6. Innovation covariance
            diff_y = y_sigma - y_pred
            Pyy = np.sum(self.Wc * diff_y * diff_y) + self.R[i]
            
            # 7. Cross-covariance between weights and measurements
            Pxy = np.zeros(self.num_features)
            for j in range(self.n_sigma):
                Pxy += self.Wc[j] * (sigma_points_pred[j] - w_pred) * diff_y[j]
            
            # 8. Kalman gain
            K = Pxy / (Pyy + 1e-12)
            
            # 9. Update weights with innovation
            innovation = chi_kp1[i] - y_pred
            self.weights[i] = w_pred + self.eta * K * innovation
            
            # 10. Update covariance
            P_update = P_pred - np.outer(K, K) * Pyy
            self.P[i] = 0.5 * (P_update + P_update.T)  # Symmetrize
    
    def get_estimate(self, chi_k, x_hat_prev, u_k):
        x_state_z = np.array([chi_k[0], chi_k[1], x_hat_prev[2]])
        z = construct_z_vector(x_state_z, u_k)
        return np.array([rhonn_predict(z, self.weights[i]) for i in range(3)])

# ==========================================
# 6. PF Trainer
# ==========================================
class PF_RHONN_Trainer:
    def __init__(self, num_neurons, num_features, n_particles, initial_weights=None):
        self.n_particles = n_particles
        self.num_features = num_features
        
        self.Q_std = 0.02
        self.R_var = 0.05
        self.ess_threshold = n_particles / 2.0
        
        self.particles = []
        self.weights_pf = []
        
        for i in range(num_neurons):
            base_w = initial_weights[i] if initial_weights is not None else np.random.randn(num_features)*0.1
            p_i = base_w + np.random.randn(n_particles, num_features) * 0.1
            self.particles.append(p_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def update(self, chi_kp1, chi_k, u_k, x_hat_prev):
        x_state_z = np.array([chi_k[0], chi_k[1], x_hat_prev[2]])
        z = construct_z_vector(x_state_z, u_k)
        
        for i in range(len(self.particles)):
            noise = np.random.normal(0, self.Q_std, size=self.particles[i].shape)
            self.particles[i] += noise
            
            preds = np.dot(self.particles[i], z)
            innov = chi_kp1[i] - preds
            
            log_likelihood = -np.abs(innov) / self.R_var
            likelihood = np.exp(log_likelihood)
            self.weights_pf[i] *= likelihood
            
            w_sum = np.sum(self.weights_pf[i])
            if w_sum < 1e-300:
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= w_sum
                
            ess = 1.0 / np.sum(self.weights_pf[i]**2)
            if ess < self.ess_threshold:
                self.resample(i)

    def resample(self, idx):
        weights = self.weights_pf[idx]
        particles = self.particles[idx]
        N = self.n_particles
        
        positions = (np.arange(N) + np.random.random()) / N
        indexes = np.zeros(N, 'i')
        cumulative_sum = np.cumsum(weights)
        i, j = 0, 0
        while i < N:
            if positions[i] < cumulative_sum[j]:
                indexes[i] = j
                i += 1
            else:
                j += 1
        
        self.particles[idx] = particles[indexes]
        self.weights_pf[idx] = np.ones(N) / N

    def get_estimate(self, chi_k, x_hat_prev, u_k):
        x_state_z = np.array([chi_k[0], chi_k[1], x_hat_prev[2]])
        z = construct_z_vector(x_state_z, u_k)
        
        estimates = []
        for i in range(3):
            mean_w = np.average(self.particles[i], weights=self.weights_pf[i], axis=0)
            estimates.append(rhonn_predict(z, mean_w))
            
        return np.array(estimates)

# ==========================================
# 7. Single Simulation Run
# ==========================================
def run_single_simulation(seed=None):
    if seed is not None:
        np.random.seed(seed)
    
    h_true = np.zeros((STEPS, 3))
    h_ekf = np.zeros((STEPS, 3))
    h_ukf = np.zeros((STEPS, 3))
    h_pf = np.zeros((STEPS, 3))
    
    x_true = np.zeros(3)
    x_ekf = np.zeros(3)
    x_ukf = np.zeros(3)
    x_pf = np.zeros(3)
    
    h_true[0] = x_true
    h_ekf[0] = x_ekf
    h_ukf[0] = x_ukf
    h_pf[0] = x_pf
    
    init_w = np.random.uniform(-0.1, 0.1, (NUM_NEURONS, NUM_FEATURES))
    
    ekf = EKF_RHONN_Trainer(NUM_NEURONS, NUM_FEATURES, eta=1.0, initial_weights=init_w)
    ukf = UKF_RHONN_Trainer(NUM_NEURONS, NUM_FEATURES, eta=1.0, initial_weights=init_w)
    pf = PF_RHONN_Trainer(NUM_NEURONS, NUM_FEATURES, NUM_PARTICLES, initial_weights=init_w)
    
    for k in range(STEPS - 1):
        t = k * DT
        
        v = 0.5 + 0.3 * np.sin(0.5 * t)
        w = 0.3 + 0.2 * np.cos(0.3 * t)
        u = np.array([v, w])
        
        x_next_true = plant_step(x_true, u, DT, PROCESS_NOISE_STD)
        
        ekf.update(x_next_true, x_true, u, x_ekf)
        x_next_ekf = ekf.get_estimate(x_true, x_ekf, u)
        
        ukf.update(x_next_true, x_true, u, x_ukf)
        x_next_ukf = ukf.get_estimate(x_true, x_ukf, u)
        
        pf.update(x_next_true, x_true, u, x_pf)
        x_next_pf = pf.get_estimate(x_true, x_pf, u)
        
        x_true = x_next_true
        x_ekf = x_next_ekf
        x_ukf = x_next_ukf
        x_pf = x_next_pf
        
        h_true[k+1] = x_true
        h_ekf[k+1] = x_ekf
        h_ukf[k+1] = x_ukf
        h_pf[k+1] = x_pf
    
    return h_true, h_ekf, h_ukf, h_pf

# ==========================================
# 8. Monte Carlo Analysis
# ==========================================
def run_monte_carlo():
    print(f"Running {NUM_MONTE_CARLO} Monte Carlo simulations...")
    
    mse_ekf_all = []
    mse_ukf_all = []
    mse_pf_all = []
    
    mae_ekf_all = []
    mae_ukf_all = []
    mae_pf_all = []
    
    rmse_ekf_all = []
    rmse_ukf_all = []
    rmse_pf_all = []
    
    # Per-state errors
    mse_px_ekf = []
    mse_py_ekf = []
    mse_theta_ekf = []
    
    mse_px_ukf = []
    mse_py_ukf = []
    mse_theta_ukf = []
    
    mse_px_pf = []
    mse_py_pf = []
    mse_theta_pf = []
    
    for run in range(NUM_MONTE_CARLO):
        if (run + 1) % 10 == 0:
            print(f"  Run {run + 1}/{NUM_MONTE_CARLO}")
        
        h_true, h_ekf, h_ukf, h_pf = run_single_simulation(seed=run)
        
        # Overall MSE
        mse_ekf = np.mean((h_true - h_ekf)**2)
        mse_ukf = np.mean((h_true - h_ukf)**2)
        mse_pf = np.mean((h_true - h_pf)**2)
        
        mse_ekf_all.append(mse_ekf)
        mse_ukf_all.append(mse_ukf)
        mse_pf_all.append(mse_pf)
        
        # MAE
        mae_ekf_all.append(np.mean(np.abs(h_true - h_ekf)))
        mae_ukf_all.append(np.mean(np.abs(h_true - h_ukf)))
        mae_pf_all.append(np.mean(np.abs(h_true - h_pf)))
        
        # RMSE
        rmse_ekf_all.append(np.sqrt(mse_ekf))
        rmse_ukf_all.append(np.sqrt(mse_ukf))
        rmse_pf_all.append(np.sqrt(mse_pf))
        
        # Per-state MSE
        mse_px_ekf.append(np.mean((h_true[:, 0] - h_ekf[:, 0])**2))
        mse_py_ekf.append(np.mean((h_true[:, 1] - h_ekf[:, 1])**2))
        mse_theta_ekf.append(np.mean((h_true[:, 2] - h_ekf[:, 2])**2))
        
        mse_px_ukf.append(np.mean((h_true[:, 0] - h_ukf[:, 0])**2))
        mse_py_ukf.append(np.mean((h_true[:, 1] - h_ukf[:, 1])**2))
        mse_theta_ukf.append(np.mean((h_true[:, 2] - h_ukf[:, 2])**2))
        
        mse_px_pf.append(np.mean((h_true[:, 0] - h_pf[:, 0])**2))
        mse_py_pf.append(np.mean((h_true[:, 1] - h_pf[:, 1])**2))
        mse_theta_pf.append(np.mean((h_true[:, 2] - h_pf[:, 2])**2))
    
    return {
        'mse_ekf': np.array(mse_ekf_all),
        'mse_ukf': np.array(mse_ukf_all),
        'mse_pf': np.array(mse_pf_all),
        'mae_ekf': np.array(mae_ekf_all),
        'mae_ukf': np.array(mae_ukf_all),
        'mae_pf': np.array(mae_pf_all),
        'rmse_ekf': np.array(rmse_ekf_all),
        'rmse_ukf': np.array(rmse_ukf_all),
        'rmse_pf': np.array(rmse_pf_all),
        'mse_px_ekf': np.array(mse_px_ekf),
        'mse_py_ekf': np.array(mse_py_ekf),
        'mse_theta_ekf': np.array(mse_theta_ekf),
        'mse_px_ukf': np.array(mse_px_ukf),
        'mse_py_ukf': np.array(mse_py_ukf),
        'mse_theta_ukf': np.array(mse_theta_ukf),
        'mse_px_pf': np.array(mse_px_pf),
        'mse_py_pf': np.array(mse_py_pf),
        'mse_theta_pf': np.array(mse_theta_pf),
    }

# ==========================================
# 9. Visualization Functions
# ==========================================
def plot_single_trajectory():
    """Plot 1: Single trajectory comparison"""
    print("\nGenerating single trajectory plot...")
    h_true, h_ekf, h_ukf, h_pf = run_single_simulation(seed=42)
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "2D Trajectory Comparison",
            "Position X over Time",
            "Position Y over Time",
            "Orientation θ over Time"
        ),
        specs=[[{"type": "scatter"}, {"type": "scatter"}],
               [{"type": "scatter"}, {"type": "scatter"}]],
        vertical_spacing=0.12,
        horizontal_spacing=0.10
    )
    
    # 2D Trajectory
    fig.add_trace(go.Scatter(x=h_true[:, 0], y=h_true[:, 1], mode='lines',
                            name='Sistema Real', line=dict(color='green', width=2.5)),
                 row=1, col=1)
    fig.add_trace(go.Scatter(x=h_ekf[:, 0], y=h_ekf[:, 1], mode='lines',
                            name='EKF-RHONN', line=dict(color='red', width=1.5, dash='dash')),
                 row=1, col=1)
    fig.add_trace(go.Scatter(x=h_ukf[:, 0], y=h_ukf[:, 1], mode='lines',
                            name='UKF-RHONN', line=dict(color='orange', width=1.5, dash='dashdot')),
                 row=1, col=1)
    fig.add_trace(go.Scatter(x=h_pf[:, 0], y=h_pf[:, 1], mode='lines',
                            name='PF-RHONN', line=dict(color='blue', width=1.8, dash='dot')),
                 row=1, col=1)
    
    time = np.arange(STEPS) * DT
    
    # X position
    fig.add_trace(go.Scatter(x=time, y=h_true[:, 0], mode='lines',
                            name='Real', line=dict(color='green', width=2),
                            showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=time, y=h_ekf[:, 0], mode='lines',
                            name='EKF', line=dict(color='red', width=1, dash='dash'),
                            showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=time, y=h_ukf[:, 0], mode='lines',
                            name='UKF', line=dict(color='orange', width=1, dash='dashdot'),
                            showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=time, y=h_pf[:, 0], mode='lines',
                            name='PF', line=dict(color='blue', width=1.2, dash='dot'),
                            showlegend=False), row=1, col=2)
    
    # Y position
    fig.add_trace(go.Scatter(x=time, y=h_true[:, 1], mode='lines',
                            line=dict(color='green', width=2), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=time, y=h_ekf[:, 1], mode='lines',
                            line=dict(color='red', width=1, dash='dash'), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=time, y=h_ukf[:, 1], mode='lines',
                            line=dict(color='orange', width=1, dash='dashdot'), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=time, y=h_pf[:, 1], mode='lines',
                            line=dict(color='blue', width=1.2, dash='dot'), showlegend=False), row=2, col=1)
    
    # Theta
    fig.add_trace(go.Scatter(x=time, y=h_true[:, 2], mode='lines',
                            line=dict(color='green', width=2), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=time, y=h_ekf[:, 2], mode='lines',
                            line=dict(color='red', width=1, dash='dash'), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=time, y=h_ukf[:, 2], mode='lines',
                            line=dict(color='orange', width=1, dash='dashdot'), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=time, y=h_pf[:, 2], mode='lines',
                            line=dict(color='blue', width=1.2, dash='dot'), showlegend=False), row=2, col=2)
    
    fig.update_xaxes(title_text="x [m]", row=1, col=1)
    fig.update_yaxes(title_text="y [m]", row=1, col=1)
    fig.update_xaxes(title_text="Tiempo [s]", row=1, col=2)
    fig.update_yaxes(title_text="x [m]", row=1, col=2)
    fig.update_xaxes(title_text="Tiempo [s]", row=2, col=1)
    fig.update_yaxes(title_text="y [m]", row=2, col=1)
    fig.update_xaxes(title_text="Tiempo [s]", row=2, col=2)
    fig.update_yaxes(title_text="θ [rad]", row=2, col=2)
    
    fig.update_layout(
        height=800,
        width=1200,
        title_text="Robot Diferencial: Comparación de Trayectorias (Ruido Laplaciano)",
        font=dict(size=11)
    )
    
    fig.show()
    return h_true, h_ekf, h_ukf, h_pf

def plot_errors_over_time(h_true, h_ekf, h_ukf, h_pf):
    """Plot 2: Error evolution over time"""
    print("Generating error evolution plots...")
    
    err_ekf = np.sqrt(np.sum((h_true - h_ekf)**2, axis=1))
    err_ukf = np.sqrt(np.sum((h_true - h_ukf)**2, axis=1))
    err_pf = np.sqrt(np.sum((h_true - h_pf)**2, axis=1))
    
    time = np.arange(STEPS) * DT
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(x=time, y=err_ekf, mode='lines',
                            name=f'EKF-RHONN (RMSE={np.mean(err_ekf):.4f})',
                            line=dict(color='red', width=1.5)))
    fig.add_trace(go.Scatter(x=time, y=err_ukf, mode='lines',
                            name=f'UKF-RHONN (RMSE={np.mean(err_ukf):.4f})',
                            line=dict(color='orange', width=1.5)))
    fig.add_trace(go.Scatter(x=time, y=err_pf, mode='lines',
                            name=f'PF-RHONN (RMSE={np.mean(err_pf):.4f})',
                            line=dict(color='blue', width=2)))
    
    fig.update_layout(
        title="Evolución del Error Euclidiano en el Tiempo",
        xaxis_title="Tiempo [s]",
        yaxis_title="Error Euclidiano",
        height=500,
        width=1000,
        font=dict(size=12)
    )
    
    fig.show()

def plot_monte_carlo_results(mc_results):
    """Plot 3: Monte Carlo statistical analysis"""
    print("Generating Monte Carlo analysis plots...")
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Distribución de MSE",
            "Distribución de RMSE",
            "Comparación de Métricas por Estado",
            "Box Plot: MSE Global"
        ),
        specs=[[{"type": "histogram"}, {"type": "histogram"}],
               [{"type": "bar"}, {"type": "box"}]],
        vertical_spacing=0.15,
        horizontal_spacing=0.12
    )
    
    # MSE Distribution
    fig.add_trace(go.Histogram(x=mc_results['mse_ekf'], name='EKF',
                               marker_color='red', opacity=0.6, nbinsx=20),
                 row=1, col=1)
    fig.add_trace(go.Histogram(x=mc_results['mse_ukf'], name='UKF',
                               marker_color='orange', opacity=0.6, nbinsx=20),
                 row=1, col=1)
    fig.add_trace(go.Histogram(x=mc_results['mse_pf'], name='PF',
                               marker_color='blue', opacity=0.6, nbinsx=20),
                 row=1, col=1)
    
    # RMSE Distribution
    fig.add_trace(go.Histogram(x=mc_results['rmse_ekf'], name='EKF',
                               marker_color='red', opacity=0.6, nbinsx=20,
                               showlegend=False), row=1, col=2)
    fig.add_trace(go.Histogram(x=mc_results['rmse_ukf'], name='UKF',
                               marker_color='orange', opacity=0.6, nbinsx=20,
                               showlegend=False), row=1, col=2)
    fig.add_trace(go.Histogram(x=mc_results['rmse_pf'], name='PF',
                               marker_color='blue', opacity=0.6, nbinsx=20,
                               showlegend=False), row=1, col=2)
    
    # Per-state comparison
    states = ['x', 'y', 'θ']
    x_pos = np.arange(len(states))
    width = 0.25
    
    mse_states_ekf = [np.mean(mc_results['mse_px_ekf']),
                      np.mean(mc_results['mse_py_ekf']),
                      np.mean(mc_results['mse_theta_ekf'])]
    mse_states_ukf = [np.mean(mc_results['mse_px_ukf']),
                      np.mean(mc_results['mse_py_ukf']),
                      np.mean(mc_results['mse_theta_ukf'])]
    mse_states_pf = [np.mean(mc_results['mse_px_pf']),
                     np.mean(mc_results['mse_py_pf']),
                     np.mean(mc_results['mse_theta_pf'])]
    
    fig.add_trace(go.Bar(x=states, y=mse_states_ekf, name='EKF',
                        marker_color='red', showlegend=False), row=2, col=1)
    fig.add_trace(go.Bar(x=states, y=mse_states_ukf, name='UKF',
                        marker_color='orange', showlegend=False), row=2, col=1)
    fig.add_trace(go.Bar(x=states, y=mse_states_pf, name='PF',
                        marker_color='blue', showlegend=False), row=2, col=1)
    
    # Box plots
    fig.add_trace(go.Box(y=mc_results['mse_ekf'], name='EKF',
                        marker_color='red', showlegend=False), row=2, col=2)
    fig.add_trace(go.Box(y=mc_results['mse_ukf'], name='UKF',
                        marker_color='orange', showlegend=False), row=2, col=2)
    fig.add_trace(go.Box(y=mc_results['mse_pf'], name='PF',
                        marker_color='blue', showlegend=False), row=2, col=2)
    
    fig.update_xaxes(title_text="MSE", row=1, col=1)
    fig.update_xaxes(title_text="RMSE", row=1, col=2)
    fig.update_xaxes(title_text="Estado", row=2, col=1)
    fig.update_xaxes(title_text="Algoritmo", row=2, col=2)
    fig.update_yaxes(title_text="Frecuencia", row=1, col=1)
    fig.update_yaxes(title_text="Frecuencia", row=1, col=2)
    fig.update_yaxes(title_text="MSE Promedio", row=2, col=1)
    fig.update_yaxes(title_text="MSE", row=2, col=2)
    
    fig.update_layout(
        height=900,
        width=1200,
        title_text=f"Análisis Estadístico Monte Carlo ({NUM_MONTE_CARLO} corridas)",
        barmode='group',
        font=dict(size=11)
    )
    
    fig.show()

def print_statistical_summary(mc_results):
    """Print comprehensive statistical summary"""
    print("\n" + "="*80)
    print("RESUMEN ESTADÍSTICO - ANÁLISIS MONTE CARLO")
    print("="*80)
    
    metrics = ['mse', 'rmse', 'mae']
    algorithms = ['ekf', 'ukf', 'pf']
    alg_names = {'ekf': 'EKF-RHONN', 'ukf': 'UKF-RHONN', 'pf': 'PF-RHONN'}
    
    for metric in metrics:
        print(f"\n{metric.upper()}:")
        print("-" * 80)
        for alg in algorithms:
            key = f'{metric}_{alg}'
            data = mc_results[key]
            print(f"{alg_names[alg]:12s} | Media: {np.mean(data):.6f} | "
                  f"Desv.Est: {np.std(data):.6f} | Min: {np.min(data):.6f} | "
                  f"Max: {np.max(data):.6f}")
    
    print("\n" + "-"*80)
    print("MSE POR ESTADO:")
    print("-"*80)
    states = ['px', 'py', 'theta']
    state_names = {'px': 'Posición X', 'py': 'Posición Y', 'theta': 'Orientación θ'}
    
    for state in states:
        print(f"\n{state_names[state]}:")
        for alg in algorithms:
            key = f'mse_{state}_{alg}'
            data = mc_results[key]
            print(f"  {alg_names[alg]:12s}: {np.mean(data):.6f} ± {np.std(data):.6f}")
    
    # Statistical tests
    print("\n" + "="*80)
    print("PRUEBAS ESTADÍSTICAS (Wilcoxon Signed-Rank Test)")
    print("="*80)
    
    # PF vs EKF
    stat_pf_ekf, p_pf_ekf = stats.wilcoxon(mc_results['mse_pf'], mc_results['mse_ekf'])
    print(f"PF vs EKF: estadístico={stat_pf_ekf:.2f}, p-valor={p_pf_ekf:.6f}")
    
    # PF vs UKF
    stat_pf_ukf, p_pf_ukf = stats.wilcoxon(mc_results['mse_pf'], mc_results['mse_ukf'])
    print(f"PF vs UKF: estadístico={stat_pf_ukf:.2f}, p-valor={p_pf_ukf:.6f}")
    
    # UKF vs EKF
    stat_ukf_ekf, p_ukf_ekf = stats.wilcoxon(mc_results['mse_ukf'], mc_results['mse_ekf'])
    print(f"UKF vs EKF: estadístico={stat_ukf_ekf:.2f}, p-valor={p_ukf_ekf:.6f}")
    
    # Improvement percentages
    print("\n" + "="*80)
    print("MEJORA PORCENTUAL (respecto a EKF)")
    print("="*80)
    
    improvement_ukf = ((np.mean(mc_results['mse_ekf']) - np.mean(mc_results['mse_ukf'])) / 
                       np.mean(mc_results['mse_ekf']) * 100)
    improvement_pf = ((np.mean(mc_results['mse_ekf']) - np.mean(mc_results['mse_pf'])) / 
                      np.mean(mc_results['mse_ekf']) * 100)
    
    print(f"UKF-RHONN: {improvement_ukf:+.2f}% (MSE)")
    print(f"PF-RHONN:  {improvement_pf:+.2f}% (MSE)")
    print("="*80 + "\n")

# ==========================================
# 10. Main Execution
# ==========================================
def main():
    print("="*80)
    print("SIMULACIÓN: Robot Diferencial con RHONN")
    print("Comparación: EKF vs UKF vs PF bajo Ruido Laplaciano")
    print("="*80)
    
    # Single trajectory plots
    h_true, h_ekf, h_ukf, h_pf = plot_single_trajectory()
    plot_errors_over_time(h_true, h_ekf, h_ukf, h_pf)
    
    # Monte Carlo analysis
    mc_results = run_monte_carlo()
    plot_monte_carlo_results(mc_results)
    print_statistical_summary(mc_results)
    
    print("\n✓ Análisis completo finalizado")

if __name__ == "__main__":
    main()

SIMULACIÓN: Robot Diferencial con RHONN
Comparación: EKF vs UKF vs PF bajo Ruido Laplaciano

Generating single trajectory plot...


Generating error evolution plots...


Running 50 Monte Carlo simulations...


KeyboardInterrupt: 